<a href="https://colab.research.google.com/github/deepak165-code/Transfer-Learning/blob/main/Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models, optimizers
from keras.layers import GlobalAveragePooling2D
from keras.optimizers import Adam
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input

import matplotlib.pyplot as plt

Load Fashion Mnist dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
print(x_train.shape)
print(y_train.shape)

print(x_test.shape)

# create validation data
x_val = x_train[-5000:]
y_val = y_train[-5000:]

x_train = x_train[:-5000]
y_train = y_train[:-5000]

print('-'*25)
print(x_val.shape)
print(y_val.shape)
print('-'*25)
print(x_train.shape)
print(y_train.shape)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(60000,)
(10000, 28, 28)
-------------------------
(5000, 28, 28)
(5000,)
-------------------------
(55000, 28, 28)
(55000,)


Define class names

In [ ]:
class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankel Boot"]

Preprocessing of fashion mnist data for VGG16

-convert img from uint8 to float32

-change chanel dimension: (28,28) -> (28,28,1)

-gray to rgb

-resize to VGG16 input size

-VGG16 preprocessing

In [ ]:
def preprocess_image(image, label):
  image = tf.cast(image, tf.float32)
  image = tf.expand_dims(image, axis = -1)
  image = tf.image.grayscale_to_rgb(image)
  image = tf.image.resize(image, [224,224])
  image = preprocess_input(image)
  return (image, label)

Create tf.data DATASET

In [ ]:
BATCH_SIZE = 32
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))


Apply Preprocessing

In [ ]:
train_dataset = (
    train_dataset
    .shuffle(10000)
    .map(preprocess_image)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
test_dataset = (
    test_dataset
    .map(preprocess_image)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_dataset = (
    val_dataset
    .map(preprocess_image)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Load VGG16

In [ ]:
base_model = VGG16(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# Freeze VGG16

In [ ]:
base_model.trainable = False

# Create our classification model


In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation = 'relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation = 'softmax')
])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 133,898 (523.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

compiling the model

In [ ]:
model.compile(
    optimizer = Adam(learning_rate = 0.001),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

Training the classifier

In [ ]:
history = model.fit(
    train_dataset,
    validation_data = val_dataset,
    epochs = 10
)

Epoch 1/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 344s 190ms/step - accuracy: 0.8138 - loss: 0.5555 - val_accuracy: 0.8822 - val_loss: 0.3287
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 347s 179ms/step - accuracy: 0.8618 - loss: 0.3875 - val_accuracy: 0.8898 - val_loss: 0.3057
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 324s 180ms/step - accuracy: 0.8719 - loss: 0.3552 - val_accuracy: 0.9006 - val_loss: 0.2838
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 305s 177ms/step - accuracy: 0.8775 - loss: 0.3362 - val_accuracy: 0.8948 - val_loss: 0.2934
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 340s 188ms/step - accuracy: 0.8837 - loss: 0.3225 - val_accuracy: 0.8916 - val_loss: 0.2909
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 323s 188ms/step - accuracy: 0.8859 - loss: 0.3149 - val_accuracy: 0.9022 - val_loss: 0.2763
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 308s 179ms/step - accuracy: 0.8908 - loss: 0.3027 - val_accuracy: 0.9012 - val_loss: 0.2772
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 321s 179ms/step - ac

Evaluation of model

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print(test_loss)
print(test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 62s 196ms/step - accuracy: 0.8989 - loss: 0.2967
0.29671618342399597
0.8988999724388123
